<h1>Case Study I: Cyprus</h1>
<h3>River Basin covering the full island, Hot Temperate summer dry hot summer (Csa) climate</h3>

<h2>Data Used:</h2>

- Flo1K (Barbarossa et al. 2018): streamflow raster   
- Mindat (Von Bargen et al. 2025): collected from the API in this notebook  
- HydroSHEDS (Lehrer et al. 2008): Flow Direction & conditioned DEM raster
- HydroBASINS (Lehner and Grill 2013): Basin vectors    
- Mining polygons (Tang and Werner 2023): Mining area vectors   

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
import geopandas as gpd
from data_utils import *
from classes import AMDModel
import contextily as cx
import matplotlib.colors as mcolors
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
new_model_initrun = True

<h1>Configuration</h1>

In [ ]:
time_first = "2005"
time_last = "2015"
region = "Cyprus"
hydrobasins_region_code = "eu"
basins_iloc = [47, 48]
case_study_nr = "CSIII"
hybas_ids_of_interest = [2040000010]
metre_crs = "EPSG:6312" #(Cyprus)
step_size = (52, "W")
f = 1e-6
alpha_s = 1e-5
A_s_ratio = 0.5
buffer_capacity = 0.5

<h2>Collect Mindat data </h2>
<p> Mindat data is used to filter if the mining polygons from Tang and Werner 2023, actually have the mineral of interest. For this case study the mineral is set as pyrite using keyword aguments: <b>material_name: "pyrite", mineral_id: 3314, material_strings: "(Fe|S)</b>.</p>

<p> The mindat_collector() function relies on the fact that you have access to the mindat API with an API key saved in the repository. The name of the file can be changed through the keyword argument: <b>mindat_api_str</b>.

In [ ]:
mindat_collector(region)

<h2>Flo1K preperation</h2>
<p>Note that the file input and output locations can be changed through the use of keyword arguments, one especially of note is the <b>date</b> keyword argument which takes a pandas Timestamp object as input, only the year argument of this Timestamp should be changed, while the month and day should stay at 1, as the Flo1K dataset works on yearly data set to <b>year/01/01</b>.</p>

In [ ]:
basins = gpd.read_file(f"../data/hybas_{hydrobasins_region_code}_lev04/hybas_{hydrobasins_region_code}_lev04_v1c.shp")

matching_rows = basins[basins["HYBAS_ID"].isin(hybas_ids_of_interest)]
if matching_rows.empty:
    print("No matching HYBAS_ID found.")
    print("Sample HYBAS_ID values:", basins["HYBAS_ID"].head(20).tolist())
    raise ValueError(f"HYBAS_ID not found: {hybas_ids_of_interest}")
iloc_indices = matching_rows.index.tolist()
iloc_tuple = tuple(iloc_indices)
aoi = basins.iloc[iloc_indices[0]:iloc_indices[-1] + 1]
basins_iloc[0] = iloc_indices[0]
basins_iloc[1] = iloc_indices[-1] + 1
aoi = basins.iloc[basins_iloc[0]: basins_iloc[1]]
ax = aoi.boundary.plot(figsize=(8, 8), edgecolor = "k")
cx.add_basemap(ax, crs = aoi.crs)

In [ ]:
from shapely.geometry import box

# define Cyprus bounding box (minx, miny, maxx, maxy)
cyprus_bbox = box(32.3, 34.6, 34.6, 35.7)

# clip the AOI to Cyprus only
aoi = gpd.clip(aoi, cyprus_bbox)

# re-plot to verify
ax = aoi.boundary.plot(figsize=(8, 8), edgecolor="k")
cx.add_basemap(ax, crs=aoi.crs)
plt.show()

In [ ]:
time_array = np.ndarray(2, dtype = "datetime64[D]")
time_array[0] = np.datetime64(f"{time_first}-01-01")
time_array[-1] = np.datetime64(f"{time_last}-01-01")
flo1k_prep(basins_iloc = basins_iloc, date= time_array, output_path=f"../data/flo_{case_study_nr}_", basins_path= f"../data/hybas_{hydrobasins_region_code}_lev04/hybas_{hydrobasins_region_code}_lev04_v1c.shp", aoi = aoi)

In [ ]:
flo_aoi_date = xr.open_dataset(f"../data/flo_{case_study_nr}_{time_first}-{time_last}.nc")
ax = aoi.boundary.plot(figsize=(8, 8), edgecolor = "k")
flo_aoi_date_plot = flo_aoi_date["qav"].where(flo_aoi_date["qav"] > 0)
flo_aoi_date_plot = flo_aoi_date_plot.isel(time = 0)
norm = mcolors.LogNorm(vmin = flo_aoi_date_plot.min(), vmax = flo_aoi_date_plot.max())
flo_aoi_date_plot.plot(ax = ax, cmap = "Blues", norm = norm, cbar_kwargs = {"label": "Annual Streamflow m3/s"})
cx.add_basemap(ax, crs = aoi.crs)
plt.title("");

<h2>Flow Direction</h2>
<p>The model requires each cell in the raster to have an unique cell ID (ID), the IDs of where the flow goes to next (outID), and if the cell is the most upstream (source). To create this the HydroSHEDS hydir data is used </p>

In [ ]:
hydir_ds = xr.open_dataset(f"../data/hyd_{hydrobasins_region_code}_dir_30s/hyd_{hydrobasins_region_code}_dir_30s.tif")
hydir_ds_copy = hydir_ds.copy()
hydir_ds = hydir_ds.rename({"band_data": "hydir"})
hydir_ds_ids = hydir_IDs(hydir_ds, aoi)

In [ ]:
ax = hydir_ds_ids["source"].plot(cbar_kwargs = {"label": "Source (1:True/0:False)"})
plt.title("")
plt.show()
plt.close()

In [ ]:
flw_da = rioxarray.open_rasterio(f"../data/hyd_{hydrobasins_region_code}_dir_30s/hyd_{hydrobasins_region_code}_dir_30s.tif")
flo_aoi_date = flo_aoi_date.rio.write_crs("EPSG:4326")
flw_da_align = flw_da.rio.reproject_match(flo_aoi_date)

flwdir = flw_da_align.values[0]
transform = flw_da_align.rio.transform()
crs = flw_da_align.rio.crs
latlon = crs.to_epsg() == 4326
ref = flo_aoi_date
flwdir
flw = pyflwdir.from_array(
    flwdir, ftype="d8", transform=transform, latlon=latlon, cache=True
)
Q = flo_aoi_date["qav"].values
for t in range(len(flo_aoi_date.time)):
    Q2d = Q[t, :, :].copy()
    
    Q2d = np.where(np.isnan(Q2d), 0.0, Q2d)
    q_accumulated = flw.accuflux(
        data=Q2d,
        nodata=-9999.0
    )
    
    Q[t, :, :] = q_accumulated
flo_aoi_date["qav"].values = Q

<h2>Mining Polygons and Mineral Occurences</h2>
<p>Because no good dataset exists where mining areas are matched with what ores are present this matching has to be here. The principle is as follows, the mining polygons (Tang and Werner 2013) are first converted to raster datasets (to align with the other datasets) with a bolean array of mine or no mine. The mindat mineral occurences are points on a map, but all have some error associated with them, thus the mineral points get a buffer of around 5km/0.045 degrees. If a mine cell intersects with the buffer of a mineral occurence the mine is kept, if not the mine likely does not have the mineral and is dropped. </p>

In [ ]:
vector_rasterisation(flo1k_path=f"../data/flo_{case_study_nr}_{time_first}-{time_last}.nc")

mines_raster = xr.open_dataset("../data/mines_raster.tif")
mineral_points = gpd.read_file(f"../data/mindat_data/{region}_pyrite.csv")
mineral_points = gpd.points_from_xy(mineral_points["longitude"], mineral_points["latitude"], crs = "EPSG:4326")
mineral_points = gpd.GeoDataFrame(mineral_points).set_geometry(col = 0)

filtered_mines_raster = filter_mines_with_buffer(mines_raster, mineral_points, metre_crs)

<h2>Merging Datasets</h2>
<p>For the model a single dataset is used, thus the different datasets are merged using xarray.merge.</p>   
<p>Note that labels, descriptions and units are changed in the function cleanup_and_metadata() in the funcs file </p>

In [ ]:
# convert bools to ints
hydir_ds_ids_num = bool_to_int(hydir_ds_ids.copy())   

# reproject
flo_aoi_date = flo_aoi_date.rio.write_crs("EPSG:4326")
ref = flo_aoi_date

hydir_aligned = hydir_ds_ids_num.rio.reproject_match(ref)
mines_aligned = filtered_mines_raster.rio.reproject_match(ref)
# merge
ds_combined = xr.merge([ref, hydir_aligned, mines_aligned])

# cleanup
ds = cleanup_and_metadata(ds_combined)

<h2>Adding Time dimension </h2>
<p>Although the Flo1K dataset already has a time dimension of years 1960-2015, a more discrete timestep is needed for the model, with the function add_time (see funcs.py) this more discrete time dimension is added.</p>

In [ ]:
ds = add_time(ds, step_size[0], step_size[1])

<h2>Estimating Ore amounts</h2>
<p>For each cell of "mines", a reactive ore amount must be estimated. This is done using the equation:       


$$
reactive_{ore} = 27000f
$$

In [ ]:
ds = estimate_ore(ds, f) 

In [ ]:
ds

<h2>Modelling</h2>

In [ ]:
if new_model_initrun:
    model = AMDModel(ds, "week", output_path= f"../data/validation data/AMDFLOW_{case_study_nr}_{time_first}-{time_last}_W.nc",
                     alpha_s = alpha_s, A_s_ratio= A_s_ratio, buffer_capacity = buffer_capacity)

In [ ]:
if new_model_initrun:
    model.run()

<h2>Validation

In [ ]:
if new_model_initrun:
    model.dataset = ds

In [ ]:
from val_utils import *
case_study_nr = "CSIII"
times = ("1960", "2015")
hydrobasins_region_code = "eu"
caravan_path = "../data/validation data/Caravan-Qual_lite.zarr"
amd_path = f"../data/validation data/AMDFLOW_{case_study_nr}_{times[0]}-{times[1]}_W.nc"
output_path = f"../data/validation data/{case_study_nr}/"
acc_path = f"../data/validation data/hyd_{hydrobasins_region_code}_acc_30s.tif"
rivers_path = f"../data/validation data/HydroRIVERS_v10_{hydrobasins_region_code}_shp/HydroRIVERS_v10_{hydrobasins_region_code}.shp"

utm_crs = "EPSG:6312" #"EPSG:6312" (Cyprus),  "EPSG:24378" (India) "EPSG: 3761" #(Canada)
resample_freq = "W"
var_map = {
    "pH": "pH",
    "Fe-Dis": ["ferrous_iron", "ferric_iron"],
    "Fe-Tot": ["ferrous_iron", "ferric_iron", "iron_III_hydroxide"],
}

print(f"Validating AMDFLOW Case Study {case_study_nr} ({times[0]}–{times[1]}),")
print("against Caravan-Qual Lite using HydroSHEDS network + area ratio snapping.")

# load everything
amd, caravan, acc_array, acc_transform, acc_nodata, rivers, cell_area_func = load_datasets(
    amd_path, caravan_path, acc_path, rivers_path
)

candidates = wqms_stations_domain_filter(amd, caravan)

# get masks
(valid_ilat, valid_ilon, valid_lat, valid_lon,
    iron_ilat, iron_ilon, iron_lat, iron_lon,
    valid_mask, iron_mask) = valid_masking(amd)

# build river graph
river_graph = build_river_graph(rivers)

# 2D lat/lon grids
lat_vals = amd.lat.values
lon_vals = amd.lon.values
amd_lat_2d, amd_lon_2d = np.meshgrid(lat_vals, lon_vals, indexing="ij")

# upstream areas from ACC
uparea_dict = assign_uparea_from_acc(amd_lat_2d, amd_lon_2d, valid_mask,
                                        acc_array, acc_transform, acc_nodata, cell_area_func)

# snap AMD cells to river network (once)
cell_to_river = snap_cells_to_river(amd_lat_2d, amd_lon_2d, valid_mask, rivers, utm_crs)
print(f"Cells snapped to river: {len(cell_to_river)}")
# snap stations for pH mask
matches_ph = snap_stations_hydrosheds(
    candidates, amd_lat_2d, amd_lon_2d, valid_mask,
    uparea_dict, cell_to_river, rivers, river_graph, utm_crs,
    max_network_km=5.0, area_ratio_min=0.5, area_ratio_max=2.0
)

# snap stations for iron mask
iron_cells = [(ilat, ilon) for ilat, ilon in zip(*np.where(iron_mask))]
iron_uparea_dict = {k: uparea_dict.get(k, np.nan) for k in iron_cells}
iron_cell_to_river = {k: v for k, v in cell_to_river.items() if k in iron_cells}

matches_iron = snap_stations_hydrosheds(
    candidates, amd_lat_2d, amd_lon_2d, iron_mask,
    iron_uparea_dict, iron_cell_to_river, rivers, river_graph, utm_crs,
    max_network_km=5.0, area_ratio_min=0.5, area_ratio_max=2.0
)

print(f"  pH mask:   {len(matches_ph)} stations matched")
print(f"  Iron mask: {len(matches_iron)} stations matched")

# run full validation
all_results = full_run(var_map, matches_ph, matches_iron, amd, caravan,
                        min_paired_obs=3, resample_freq=resample_freq)

for var, ts in all_results.items():
    print(f"\n── {var} ──────────────────────────────────────────")
    metrics = validation_metrics(ts)
    
    # add coordinates for matched stations (pH and iron may differ)
    if var in ["Fe-Dis", "Fe-Tot"]:
        match_df = matches_iron
    else:
        match_df = matches_ph
    
    coords = match_df[["wqms_id", "cell_lat", "cell_lon"]].set_index("wqms_id")
    results_df = metrics.join(coords)
    
    os.makedirs(output_path, exist_ok=True)
    results_df.to_csv(f"{output_path}metrics_{var}.csv")
    print(f"  Saved → {output_path}metrics_{var}.csv\n  With {len(results_df)} matched stations and their metrics.")

In [ ]:
output = xr.open_dataset(f"../data/validation data/AMDFLOW_{case_study_nr}_{time_first}-{time_last}_W.nc")

In [ ]:
ferric_iron = output["ferric_iron"]
ferrous_iron = output["ferrous_iron"]
sulphate = output["sulphate"]
hydron = output["hydrogen_ion"]
iron_III_hydroxide = output["iron_III_hydroxide"]
ph = output["pH"]
bedload_storage = output["bedload_storage"]
output.close()

In [ ]:
validation_ph = pd.read_csv(f"../data/validation data/{case_study_nr}/Metrics_pH.csv").dropna()
display(validation_ph.sort_values("KGE", ascending = False))
try:
    best_nse_ph = validation_ph.sort_values("KGE", ascending = False).iloc[0]
except:
    pass

In [ ]:
validation_fe_dis = pd.read_csv(f"../data/validation data/{case_study_nr}/Metrics_Fe-Dis.csv")
display(validation_fe_dis.sort_values("KGE", ascending = False))
try:
    best_nse_fe_dis = validation_fe_dis.sort_values("KGE", ascending = False).iloc[0]
except:
    pass

In [ ]:
validation_fe_tot = pd.read_csv(f"../data/validation data/{case_study_nr}/Metrics_Fe-Tot.csv")
display(validation_fe_tot.sort_values("R", ascending = False))
try:
    best_nse_fe_tot = validation_fe_tot.sort_values("R", ascending=False).iloc[0]
except:
    pass

In [ ]:
caravan = xr.open_dataset("../data/validation data/Caravan-Qual_lite.zarr", engine="zarr",
                               chunks={})

In [ ]:
obs_fe_tot = caravan.sel(wqms_id = best_nse_fe_tot["wqms_id"])["Fe-Tot"]
obs_ph = caravan.sel(wqms_id = best_nse_ph["wqms_id"])["pH"]

In [ ]:
model_fe_tot = ferric_iron.sel(lat = best_nse_fe_tot["cell_lat"], lon = best_nse_fe_tot["cell_lon"]) + \
        ferrous_iron.sel(lat = best_nse_fe_tot["cell_lat"], lon = best_nse_fe_tot["cell_lon"]) + \
        iron_III_hydroxide.sel(lat = best_nse_fe_tot["cell_lat"], lon = best_nse_fe_tot["cell_lon"])

model_ph = ph.sel(lat = best_nse_ph["cell_lat"], lon = best_nse_ph["cell_lon"])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(obs_fe_tot.time, obs_fe_tot.values, label = "Observed Fe-Tot", marker = "o", color = "orange")
model_fe_tot.plot(ax = ax, label = "Modeled Fe-Tot")
ax.set_title("")
# ax2 = ax.twinx()
# Q = model.dataset["Q"].sel(lat = best_nse_fe_tot["cell_lat"], lon = best_nse_fe_tot["cell_lon"], method = "nearest")
# # volume = Q / model.v * model.dx * 1000
# ax2.plot(volume.time, volume.values, linestyle = "--", color = "k", alpha = 0.5)
# ax2.set_title("")
# ax2.set_ylabel("volume (L)")
plt.legend()
plt.title("")
plt.title(f"Fe-Tot at WQMS ID {best_nse_fe_tot["wqms_id"]} (R-squared {best_nse_fe_tot["R"]:.2f})")
plt.xlabel("Time")
ax.set_ylabel("Fe-Tot (microgram/L)");

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(obs_ph.time, obs_ph.values, label = "Observed pH", marker = "o", color = "orange")
model_ph.plot(ax = ax, label = "Modeled pH")
ax.set_title("")
# ax2 = ax.twinx()
# Q = model.dataset["Q"].sel(lat = best_nse_ph["cell_lat"], lon = best_nse_ph["cell_lon"], method = "nearest")
# volume = Q / model.v * model.dx * 1000
# ax2.plot(volume.time, volume.values, linestyle = "--", color = "k", alpha = 0.5)
# ax2.set_title("")
# ax2.set_ylabel("volume (L)")
plt.legend()
plt.title(f"pH at WQMS ID {best_nse_ph["wqms_id"]} (KGE: {best_nse_ph["KGE"]:.2f})")
plt.xlabel("Time")
ax.set_ylabel("pH");

In [ ]:
fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(nrows = 1, ncols = 5, figsize = (20, 6))
ax1.violinplot(validation_ph["KGE"], showmedians = True)
ax1.set_ylabel("KGE")
ax2.violinplot(validation_ph["NSE"], showmedians = True)
ax2.set_ylabel("NSE")
ax3.violinplot(validation_ph["R"], showmedians = True)
ax3.set_ylabel("R")
ax4.violinplot(validation_ph["RMSE"], showmedians = True)
ax4.set_ylabel("RMSE")
ax5.violinplot(validation_ph["bias"], showmedians = True)
ax5.set_ylabel("bias")
plt.tight_layout();

In [ ]:
fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(nrows = 1, ncols = 5, figsize = (20, 6))
ax1.violinplot(validation_fe_tot["KGE"], showmedians = True)
ax1.set_ylabel("KGE")
ax2.violinplot(validation_fe_tot["NSE"], showmedians = True)
ax2.set_ylabel("NSE")
ax3.violinplot(validation_fe_tot["R"], showmedians = True)
ax3.set_ylabel("R")
ax4.violinplot(validation_fe_tot["RMSE"], showmedians = True)
ax4.set_ylabel("RMSE")
ax5.violinplot(validation_fe_tot["bias"], showmedians = True)
ax5.set_ylabel("bias")
plt.tight_layout();

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ferrous_iron.where(ferrous_iron > 0).isel(time = [-1]).plot(ax = ax)
aoi.plot(ax = ax,facecolor="None",  edgecolor= "k")
cx.add_basemap(ax, crs = aoi.crs)



In [ ]:
model.dataset["Q"].sel(lat = best_nse_fe_tot["cell_lat"], lon = best_nse_fe_tot["cell_lon"], method = "nearest").plot(linestyle = "--", color = "k", alpha = 0.5)

In [ ]:
ferric_best = ferric_iron.sel(lat = best_nse_fe_tot["cell_lat"], lon = best_nse_fe_tot["cell_lon"])
ferrous_best = ferrous_iron.sel(lat = best_nse_fe_tot["cell_lat"], lon = best_nse_fe_tot["cell_lon"])
hydroxide_best = iron_III_hydroxide.sel(lat = best_nse_fe_tot["cell_lat"], lon = best_nse_fe_tot["cell_lon"])

fig, ax = plt.subplots()

ax.plot(ferric_best.time, ferric_best.values, label = "ferric", linestyle = ":")
ax.plot(ferrous_best.time, ferrous_best.values + ferric_best, label = "ferrous", color = "r", linestyle = "-.")
ax.plot(hydroxide_best.time, hydroxide_best.values + ferric_best + ferrous_best, label = "hydroxide", color = "g", linestyle = "--", alpha = 0.5)
plt.legend()

In [ ]:
model_fe_tot.plot(label = "Modeled Fe-Tot")

In [ ]:
arr = ph.values
clean_arr = arr[~np.isnan(arr)]
lowest = clean_arr.min() if len(clean_arr) > 0 else np.nan
print(lowest)

In [ ]:

columns = ["KGE", "NSE", "R", "bias"]

# Create summary statistics for each metric
summary_dict = {}
for metric in columns:
    summary_dict[metric] = {
        "Fe-Tot Mean": validation_fe_tot[metric].mean(),
        "Fe-Tot Min": validation_fe_tot[metric].min(),
        "Fe-Tot Max": validation_fe_tot[metric].max(),
        "pH Mean": validation_ph[metric].mean(),
        "pH Min": validation_ph[metric].min(),
        "pH Max": validation_ph[metric].max(),
    }

summary_df = pd.DataFrame(summary_dict).T
summary_df

In [ ]:
var = ferric_iron
fig, ax = plt.subplots(figsize=(12, 6))
var.where(var > 0).isel(time = [-1]).plot(ax = ax)
aoi.plot(ax = ax,facecolor="None",  edgecolor= "k")


In [ ]:
ferric_iron.isel(time = -1).values.max()

In [ ]:
model._ore_np[model._ore_np > 0].min()

In [ ]:
arr = ds["ore"].values
clean_arr = arr[~np.isnan(arr)]
clean_arr.max()